<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/08_external_and_gradcam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unseen generators and Grad-CAM

Manuscript sections 3.8.2 (external generalization) and 3.7.3 / 3.8.3 (Grad-CAM).

**Why not Midjourney and Stable Diffusion?** Both are already in the training data (Superpotato9: DALL·E and Midjourney; AI-ArtBench:
Stable Diffusion and Latent Diffusion), so they can't test unseen generators. This uses **WikiArt_VLM** (Fu et al., 2026,
[Zenodo 10.5281/zenodo.17880956](https://zenodo.org/records/17880956), CC BY 4.0): WikiArt paintings and, for each, an image generated from
a caption of it by **FLUX.1-dev** and **F-Lite** (both unseen) and **Stable Diffusion 3.5** (a newer generation of a seen family).
Sampling is paired where possible: each sampled painting appears once as a generated image and once as the original, so subject matter
can't separate the classes. Images are harmonized exactly like the training data, and any that are near-duplicates of a training image
are removed (AI-ArtBench's human art also comes from WikiArt).

Then the single-split ResNet-50 and EfficientNet-B0 from notebook 02 go through the full evaluation (all four systems, all nine conditions,
with and without the watermark), and Grad-CAM heatmaps and attention-stability numbers are produced.

**Runtime > Change runtime type > T4 GPU.** The one-time download is about 36 GB to Colab's own disk (often 20-60 minutes, depending on
Zenodo); the prepared set (~1 GB) is saved to Drive, so later sessions skip it. Evaluation about 1 hour, Grad-CAM about 10 minutes.

## 1. Get the code

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e ".[train,provenance,dev]"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Restore the training data and link the results folders on Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
D = "/content/drive/MyDrive/thesis-data"
!unzip -qo $D/harmonized.zip -d .
for d in ["models", "watermark", "evaluation", "gradcam"]:
    !mkdir -p $D/$d && rm -rf outputs/$d && ln -s $D/$d outputs/$d
ARCHS = ["resnet50", "efficientnet_b0"]
missing = [f"outputs/models/{a}/{f}" for a in ARCHS for f in ("best.pt", "calibration.json")] + \
          [p for p in ["outputs/watermark/threshold.json"] if not os.path.exists(p)]
missing = [p for p in missing if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"missing {missing}: run notebooks 02 and 03 first")
print("models and watermark threshold found")

## 3. Get the external set
If `thesis-data/external.zip` already exists (a previous session), it is restored and the download is skipped.
Otherwise the four WikiArt_VLM archives are downloaded to Colab's disk (`wget -c` resumes an interrupted download if you rerun the cell).

In [ ]:
EXT_ZIP = f"{D}/external.zip"
VLM = "/content/wikiart_vlm"
NEED_DOWNLOAD = not os.path.exists(EXT_ZIP)
if NEED_DOWNLOAD:
    !mkdir -p $VLM && df -h /content | tail -1
    for name in ["Original.zip", "FLUX.zip", "F-Lite.zip", "Stable-Diffusion.zip"]:
        !wget -c -q --show-progress -O $VLM/$name "https://zenodo.org/records/17880956/files/$name?download=1"
    !ls -lh $VLM
else:
    !unzip -qo $EXT_ZIP -d .
    print("restored the prepared external set from Drive")

In [ ]:
# What's inside the archives, and whether generated files can be matched to their originals by name
if NEED_DOWNLOAD:
    !python scripts/prepare_external.py --zips $VLM --inspect

In [ ]:
if NEED_DOWNLOAD:
    !python scripts/prepare_external.py --zips $VLM --per-generator 1500
    !zip -qr $EXT_ZIP data/external && ls -lh $EXT_ZIP
    !rm -rf $VLM   # free the 36 GB
import json
summary = json.load(open("data/external/external_summary.json"))
print(json.dumps(summary, indent=1))
!mkdir -p outputs/evaluation/external && cp data/external/external_summary.json outputs/evaluation/external/

## 4. Evaluate on the unseen generators
Same pipeline as notebook 05, on the external set: with the watermark, then the no-watermark ablation (the CNN's own generalization).

In [ ]:
for arch in ARCHS:
    !python scripts/evaluate.py --arch $arch --split-csv data/external/external.csv --out outputs/evaluation/external
    !python scripts/evaluate.py --arch $arch --split-csv data/external/external.csv --out outputs/evaluation/external --no-watermark
!python scripts/external_report.py

## 5. Results
**Per generator:** each generator's AI images together with all the human paintings (recall = share of that generator's images flagged as AI).
**vs in-distribution:** the same metric on the original test split (notebook 05) and on the external set, and the change.
The no-watermark rows are the cleanest view of the CNN's generalization; the watermarked rows show what fusion does with it.

In [ ]:
import pandas as pd
pd.set_option("display.width", 220)
g = pd.read_csv("outputs/evaluation/external/report/by_generator.csv")
v = pd.read_csv("outputs/evaluation/external/report/vs_in_distribution.csv")
for run in g.run.unique():
    r = g[(g.run == run) & (g.system.isin(["cnn_only", "multi_signal"]))]
    order = r.condition.unique()
    print(f"\n{run}: recall (share of each generator's images flagged as AI)")
    display(r.pivot_table(index="condition", columns=["system", "generator"], values="recall").loc[order].round(3))
    print(f"{run}: false-positive rate on the human paintings")
    display(r[r.generator == "all"].pivot(index="condition", columns="system", values="fpr").loc[order].round(3))
    t = v[(v.run == run) & (v.system.isin(["cnn_only", "multi_signal"]))]
    if t.empty:
        print(f"{run}: no in-distribution results to compare with (run notebook 05 sections 4 and 7)")
        continue
    print(f"{run}: accuracy and F1, in-distribution vs external")
    display(t.pivot(index="condition", columns="system",
                    values=["accuracy_in_distribution", "accuracy_external", "f1_in_distribution", "f1_external"]).loc[order].round(3))

## 6. Grad-CAM
Heatmaps of where each CNN looks (last convolutional block), for correct and wrong examples of each class, on the clean image and after
JPEG q30 and 0.5× downscaling; the effect of the watermark on the CNN's attention; and how much of the top-20% attention region
survives each transformation (IoU with the clean heatmap) on 300 images per class.

In [ ]:
for arch in ARCHS:
    !python scripts/gradcam.py --arch $arch

In [ ]:
from IPython.display import Image as Show, display as show
for arch in ARCHS:
    for name in ["examples_in_distribution", "examples_external", "watermark"]:
        print(arch, name)
        show(Show(f"outputs/gradcam/{arch}/{name}.png", width=900))
    s = pd.read_csv(f"outputs/gradcam/{arch}/stability.csv")
    print(f"{arch}: attention stability (IoU of top-20% region with the clean heatmap) and prediction flips")
    display(s.pivot_table(index="comparison", columns=["set", "class"], values="iou_top20_mean").round(3))
    display(s.pivot_table(index="comparison", columns=["set", "class"], values="prediction_flip_rate").round(3))

## 7. Save it
File > Save a copy in GitHub. Results are on Drive (`thesis-data/evaluation/external/`, `thesis-data/gradcam/`) and are collected by notebook 06.